# 採用 current model の再現

この notebook は、ルート `README.md` と同じ正式 CLI を順に呼びます。旧 pipeline や Gaussian 計算は実行しません。すべて未実行の状態で配布しています。リポジトリのルートから kernel を起動してください。

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_ROOT = Path.cwd().resolve()
assert (REPO_ROOT / "libs" / "current_model.py").is_file(), (
    "Start this notebook from the repository root."
)
WORKERS = min(4, os.cpu_count() or 1)  # 1-20 の範囲で変更可能
RUN_ENV = os.environ.copy()
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    RUN_ENV[variable] = "1"

def run_cli(*arguments: str) -> None:
    subprocess.run(
        [sys.executable, *arguments],
        cwd=REPO_ROOT,
        env=RUN_ENV,
        check=True,
    )

## 1. 凍結入力の検証

manifest の byte 数・SHA-256 と portable input の構造を確認します。

In [ ]:
run_cli("libs/current_model.py", "--verify-inputs-only")

## 2. 完全 nested LOOCV

83 outer fold をすべて再計算します。`--skip-nested` は使用しません。Git 対象外の表示用 contribution cube のみ省略します。

In [ ]:
run_cli(
    "libs/current_model.py",
    "--workers",
    str(WORKERS),
    "--no-excel-refresh",
    "--skip-contribution-cubes",
)

## 3. 空間解析

83 outer model の係数を再構築し、空間寄与の補助表と確定図を更新します。

In [ ]:
run_cli(
    "libs/analyze_current_model_spatial_contributions.py",
    "--workers",
    str(WORKERS),
    "--no-excel-refresh",
)

## 4. 保存結果の照合

凍結 package、191 × 321 feature matrix、`summary.csv`、83 outer prediction の指標を独立に照合します。

In [ ]:
run_cli("scripts/verify_reproduction.py")